# Phase 6 — Health Indicators for the IGBT data

## What is a Health Indicator (HI)?

A **Health Indicator** is a measured or derived quantity intended to track a system's health state over time. It is more than a feature with a trend: a credible HI should have a plausible link to a degradation mechanism, respond consistently as damage accumulates, be sufficiently monotonic and sensitive, have manageable noise, and remain comparable under the operating and measurement conditions in which it is used. Its direction need not always be increasing, but the direction and meaning should be physically justified.

A candidate can look attractive statistically and still be a poor HI. A load, temperature set-point, instrument drift, acquisition setting, or file boundary can make a feature trend without device damage. Validation needs independent physical evidence (for example, a documented degradation/failure event or a calibrated electrical characterization tied to the same device and aging history).

**Goal of this lesson:** compare several plausible descriptors from the Phase 5 transient table using per-device trend, monotonicity, variability, and acquisition-stability checks. We will not create a health score, set a failure threshold, claim causation, or select a single "best" HI.

The recorded `timeEpoch` is used only as a **within-device recorded-time proxy**. It is not a cycle count, cumulative thermal dose, independently verified aging variable, or EOL label. Four devices and uneven, short capture sequences are not enough to prove prognostic performance.

## 1. Load the Phase 5 candidate feature table

Run [`03_feature_engineering.ipynb`](03_feature_engineering.ipynb) first. Its table has one row per transient capture and preserves the device, source file, capture time, and within-capture sample interval. We check required fields rather than silently substituting another dataset. Files whose names contain `check` are displayed for context but excluded from aging-trend calculations; that filename distinction is not itself a health label.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, theilslopes

project_root = Path.cwd()
if not (project_root / "data" / "features").is_dir():
    project_root = project_root.parent

feature_path = project_root / "data" / "features" / "transient_candidate_features.csv"
if not feature_path.is_file():
    raise FileNotFoundError(
        f"{feature_path} was not found. Run notebooks/03_feature_engineering.ipynb first."
    )

features = pd.read_csv(feature_path)
required_columns = {
    "device_folder", "source_file", "capture_index", "timeEpoch", "dt_seconds",
    "collectorEmitterVoltage_peak_to_peak", "collectorEmitterVoltage_absolute_peak",
    "collectorEmitterVoltage_rms", "gateEmitterVoltage_peak_to_peak",
    "collectorEmitterCurrentSignal_rms", "collectorEmitterCurrentSignal_peak_to_peak",
    "gateSignalVoltage_peak_to_peak",
}
missing = sorted(required_columns.difference(features.columns))
if missing:
    raise ValueError(f"Feature table is missing required columns: {missing}")

features["timeEpoch"] = pd.to_numeric(features["timeEpoch"], errors="raise")
features["dt_seconds"] = pd.to_numeric(features["dt_seconds"], errors="raise")
features["is_check_file"] = features["source_file"].str.contains("check", case=False, regex=False)
features["elapsed_recorded_hours"] = features["timeEpoch"]
for device, index in features.groupby("device_folder", observed=True).groups.items():
    device_times = features.loc[index, "timeEpoch"]
    features.loc[index, "elapsed_recorded_hours"] = (device_times - device_times.min()) / 3600

aging = features.loc[~features["is_check_file"]].copy()
if aging.empty:
    raise ValueError("No non-check transient captures are available for comparison.")

print(f"All capture rows: {len(features):,}; trend-comparison rows: {len(aging):,}")
print("Devices:", sorted(aging["device_folder"].unique()))
print("Check-file captures set aside:", int(features["is_check_file"].sum()))
print("Capture counts by device:")
display(aging.groupby("device_folder", observed=True).size().rename("aging captures"))

## 2. Choose candidates for physical reasons—not because they trend

We compare features from different measured channels because each answers a different physical question:

| Candidate | PHM rationale | Main caveat |
|---|---|---|
| Collector-emitter voltage peak-to-peak | Switching-voltage extremes could respond to electrical behavior; peak behavior is a mechanism-informed hypothesis from this experiment family. | Whole-capture range is not an aligned turn-off overshoot; it is sensitive to settings, capture length, scale, and outliers. |
| Collector-emitter voltage absolute peak | Simple extreme-amplitude descriptor of the measured switching-voltage trace. | Does not identify the switching event or separate overshoot from steady portions; units/calibration need checking. |
| Collector-emitter voltage RMS | Summarizes voltage magnitude over the whole capture. | Mixes operating state, waveform duty/shape, amplitude, and capture window; not inherently damage-sensitive. |
| Gate-emitter voltage peak-to-peak | May expose changes in gate-drive response or capture conditions. | Gate drive is also an input/control context; a change is not necessarily device degradation. |
| Collector-emitter current RMS / peak-to-peak | Current response can be electrically relevant. | Dataset notes warn of collector-current drift; these are especially useful as a measurement-quality caution, not a trusted HI. |
| Gate-command voltage peak-to-peak | Negative-control/context feature: should mostly reflect the applied drive, not accumulated device damage. | A time trend here would suggest changing settings or acquisition, undermining attribution of other trends. |

Steady-state temperature, voltage, and current were not added to the transient candidates: those are different measurement timescales and are not interchangeable with waveform features. Threshold-voltage, leakage, or breakdown measurements could be strong domain candidates in a properly aligned study, but the characterization files are not joined to these capture records here.

## 3. Define screening measures and their limitations

For each candidate and each device, we calculate:

- **Spearman correlation with recorded time**: rank-based trend association. Its absolute value is also our simple monotonicity score (0–1); the sign gives direction. This is not independent evidence of aging because recorded time is only a proxy.
- **Direction agreement**: whether devices' non-zero trend directions agree. A common direction is encouraging but can also reflect a common protocol or instrument drift.
- **Robust slope**: Theil–Sen slope per recorded hour, to show scale and direction without letting one point dominate. Different units make raw slopes incomparable across features.
- **Trend-residual scatter**: median absolute deviation of residuals from that robust straight-line trend divided by the feature IQR. Smaller values mean less scatter around this simple trend, not necessarily better health tracking. Curved or stage-wise degradation can be unfairly penalized.

These are screening summaries, not a composite score or hypothesis test. We report per-device results because a pooled correlation can be driven by between-device offsets, unequal record counts, or different operating protocols. Duplicate/nearby records and short runs also mean the usual independent-sample interpretation of a correlation is not justified.

In [ ]:
candidate_features = {
    "Collector voltage peak-to-peak": "collectorEmitterVoltage_peak_to_peak",
    "Collector voltage absolute peak": "collectorEmitterVoltage_absolute_peak",
    "Collector voltage RMS": "collectorEmitterVoltage_rms",
    "Gate-emitter voltage peak-to-peak": "gateEmitterVoltage_peak_to_peak",
    "Collector current RMS": "collectorEmitterCurrentSignal_rms",
    "Collector current peak-to-peak": "collectorEmitterCurrentSignal_peak_to_peak",
    "Gate-command voltage peak-to-peak (context)": "gateSignalVoltage_peak_to_peak",
}

def summarize_device_trend(group, feature_column):
    data = group[["elapsed_recorded_hours", feature_column]].replace([np.inf, -np.inf], np.nan).dropna()
    data = data.sort_values("elapsed_recorded_hours")
    x = data["elapsed_recorded_hours"].to_numpy(dtype=float)
    y = data[feature_column].to_numpy(dtype=float)
    if len(data) < 3 or np.unique(x).size < 2 or np.unique(y).size < 2:
        return None

    correlation = float(spearmanr(x, y).statistic)
    slope, intercept, _, _ = theilslopes(y, x)
    residual = y - (intercept + slope * x)
    iqr = float(np.subtract(*np.percentile(y, [75, 25])))
    residual_mad = float(np.median(np.abs(residual - np.median(residual))))
    relative_residual_scatter = residual_mad / iqr if iqr > 0 else np.nan
    return {
        "captures": len(data),
        "spearman_rho_time": correlation,
        "monotonicity_abs_rho": abs(correlation),
        "theil_sen_slope_per_hour": float(slope),
        "residual_MAD_over_feature_IQR": relative_residual_scatter,
        "feature_IQR": iqr,
    }

trend_rows = []
for candidate_name, feature_column in candidate_features.items():
    for device, group in aging.groupby("device_folder", observed=True, sort=True):
        summary = summarize_device_trend(group, feature_column)
        if summary is not None:
            trend_rows.append({"candidate": candidate_name, "feature": feature_column, "device": device, **summary})

device_trends = pd.DataFrame(trend_rows)
if device_trends.empty:
    raise ValueError("Not enough valid per-device observations to summarize candidate trends.")
device_trends.round(3)

### Reading the trend table

Look for a plausible and repeatable direction across devices, not just a large absolute correlation in one device. Also compare residual scatter: a smooth trend may be easy to monitor, but only if it reflects degradation rather than controlled operating changes. `feature_IQR` and slopes retain the feature's original units and scale; do not compare raw slope magnitudes between voltage and current features.

In [ ]:
device_trends.pivot(index="candidate", columns="device", values="spearman_rho_time").round(2)

The compact summary below makes cross-device disagreement visible. `direction_agreement` is the fraction of devices on the more common non-zero direction; a tie is 0.5. It is descriptive only and should not be used as a leaderboard, especially with four devices and no verified degradation labels.

In [ ]:
comparison_rows = []
for candidate, group in device_trends.groupby("candidate", observed=True, sort=True):
    direction = np.sign(group["spearman_rho_time"])
    nonzero_direction = direction[direction != 0]
    agreement = (
        nonzero_direction.value_counts(normalize=True).max()
        if len(nonzero_direction) else np.nan
    )
    comparison_rows.append({
        "candidate": candidate,
        "devices": group["device"].nunique(),
        "median_abs_spearman": group["monotonicity_abs_rho"].median(),
        "direction_agreement": agreement,
        "median_trend_residual_MAD_over_IQR": group["residual_MAD_over_feature_IQR"].median(),
    })
candidate_comparison = pd.DataFrame(comparison_rows).set_index("candidate")
candidate_comparison.round(3)

### A first reading of these data (not an HI selection)

In the current feature table, collector-voltage peak-to-peak has mixed direction: it decreases with recorded time in Devices 2 and 5 but is only weakly positive in Devices 3 and 4. Collector-current RMS decreases in all four devices (median absolute Spearman correlation is about 0.67), but the documented collector-current drift makes that apparent monotonicity a serious measurement-validity warning. Collector-voltage RMS rises in all four devices (median absolute correlation about 0.58); gate-emitter peak-to-peak also trends down, which may reflect a shared test stage or drive/capture conditions. The gate-command peak-to-peak feature has weak association with time in this table, but that does not rule out other changing conditions.

These observations are deliberately not translated into a ranking. Multiple descriptors trend, and their direction and physical interpretations differ. Without aligned stress settings and an independent damage measure, time trends cannot distinguish device degradation from protocol changes, measurement drift, or acquisition effects.

## 4. Visualize per-device trajectories

Each panel is a separate device; lines are not joined across source files. The faint `check` captures are shown only as context and are not used to estimate trends. Feature values are **not** normalized per capture or z-scored here, because amplitude differences may be physically important. Device-specific vertical scales are retained: this view is for within-device change, not comparing absolute device health.

In [ ]:
plot_candidates = [
    ("collectorEmitterVoltage_peak_to_peak", "Collector-emitter voltage: whole-capture peak-to-peak"),
    ("collectorEmitterCurrentSignal_rms", "Collector-emitter current: RMS (drift caution)"),
    ("gateSignalVoltage_peak_to_peak", "Gate-command voltage: peak-to-peak (context check)"),
]
devices = sorted(features["device_folder"].dropna().unique())
fig, axes = plt.subplots(
    len(plot_candidates), 1, figsize=(12, 3.2 * len(plot_candidates)),
    sharex=False, squeeze=False, constrained_layout=True,
)
for ax, (column, title) in zip(axes[:, 0], plot_candidates):
    for device in devices:
        device_data = features.loc[features["device_folder"] == device].sort_values("timeEpoch")
        regular = device_data.loc[~device_data["is_check_file"]]
        check = device_data.loc[device_data["is_check_file"]]
        if not regular.empty:
            ax.plot(regular["elapsed_recorded_hours"], regular[column], ".", linestyle="none", ms=3, alpha=0.7, label=device)
        if not check.empty:
            ax.scatter(check["elapsed_recorded_hours"], check[column], marker="x", s=35, alpha=0.6)
    ax.set_title(title)
    ax.set_ylabel("Recorded feature value")
    ax.grid(alpha=0.25)
    ax.legend(ncol=min(4, len(devices)), fontsize="small")
axes[-1, 0].set_xlabel("Elapsed recorded clock time within device (hours; proxy only)")
fig.suptitle("Candidate feature trajectories — points are captures, not failure stages")
plt.show()

### PHM interpretation prompts

- Is a feature moving in the same direction across devices, or do devices disagree?
- Does the gate-command context feature move too? If so, input settings or acquisition may explain part of the response.
- Does a sharp shift line up with a source-file boundary, a check capture, or a change in sampling interval?
- Could the apparent "smoothness" be an instrument drift or stress schedule rather than degradation?

A common downward current trend is particularly ambiguous here: the dataset documentation warns of collector-current drift. Reproducibility of the trend does not distinguish a shared sensor bias from a shared degradation process.

## 5. Check acquisition stability and noise

A feature should not change substantially just because the sampling interval or file segment changed. The tables below compare feature medians by `dt` and by source file within each device. These groups may also correspond to different times or test stages, so the comparisons diagnose possible confounding; they are **not** causal estimates of the effect of `dt` or proof that a feature is unstable. We express the range of group medians relative to the overall feature IQR so features with different units can be inspected side by side.

In [ ]:
stability_features = {
    "Collector voltage peak-to-peak": "collectorEmitterVoltage_peak_to_peak",
    "Collector voltage absolute peak": "collectorEmitterVoltage_absolute_peak",
    "Collector current RMS": "collectorEmitterCurrentSignal_rms",
    "Gate-command peak-to-peak": "gateSignalVoltage_peak_to_peak",
}

def grouped_median_range(group, by_column, feature_column):
    data = group[[by_column, feature_column]].replace([np.inf, -np.inf], np.nan).dropna()
    medians = data.groupby(by_column, observed=True)[feature_column].median()
    iqr = float(np.subtract(*np.percentile(data[feature_column], [75, 25])))
    if len(medians) < 2 or iqr == 0:
        return np.nan, len(medians)
    return float((medians.max() - medians.min()) / iqr), len(medians)

stability_rows = []
for candidate, column in stability_features.items():
    for device, group in aging.groupby("device_folder", observed=True, sort=True):
        dt_range, dt_groups = grouped_median_range(group, "dt_seconds", column)
        file_range, file_groups = grouped_median_range(group, "source_file", column)
        stability_rows.append({
            "candidate": candidate,
            "device": device,
            "dt_group_median_range_over_IQR": dt_range,
            "distinct_dt_groups": dt_groups,
            "source_file_median_range_over_IQR": file_range,
            "source_file_groups": file_groups,
        })

stability = pd.DataFrame(stability_rows)
stability.round(3)

### How to think about sensitivity and noise

**Sensitivity** means a measurable response to the specific degradation mechanism, not merely a large slope over clock time. Evidence would require controlled or documented degradation progression, a physically meaningful expected direction, and association with an independent damage measure. **Noise** includes measurement repeatability and unexplained variation under nominally comparable conditions. The residual scatter above is only a rough proxy; repeated healthy baselines and replicate captures at matched settings are needed to estimate true measurement noise. **Stability** means robustness to benign changes in load, temperature, sampling, instrument, and operator—not invariance to actual damage.

## 6. Compare candidates with explicit PHM criteria

Use a reasoned, mechanism-specific shortlist rather than averaging unrelated statistics into a score:

| Criterion | What would strengthen an HI claim? | What this dataset can show now |
|---|---|---|
| Physical relevance | Feature tied to a known failure/degradation mechanism, with a defined measurement protocol. | Voltage/current waveforms are relevant electrical responses; the whole-capture statistics are coarse and not event-aligned. |
| Monotonicity and trend | Consistent direction within devices over a defensible aging coordinate, preferably across repeats. | Rank trends versus recorded clock time; no verified damage coordinate, and device directions can differ. |
| Correlation with aging | Agreement with independent aging dose or inspection/failure evidence, not only elapsed time. | Only a weakly justified timestamp proxy is available in this comparison. |
| Noise / repeatability | Low variation among repeated measurements under matched conditions; confidence intervals account for dependence. | Trend-residual scatter is available, but acquisition and stress settings are not controlled here. |
| Stability / specificity | Robust under benign condition changes and negative controls, while responding to degradation. | `dt`, source-file differences, context features, and current drift can be examined as warnings, not fully controlled. |
| Sensitivity / prognostic value | Detects known damage early and usefully predicts a predeclared endpoint on held-out devices. | No validated failure endpoint or sufficiently independent device cohort for this claim. |

### Candidate-by-candidate PHM reasoning

- **Collector-emitter voltage peak-to-peak:** strongest mechanism-informed *hypothesis* among these summaries, but it is a full-capture range rather than an aligned turn-off overshoot. Device-to-device direction disagreement or source/setting sensitivity weakens the case. Keep as a candidate for event-aligned reanalysis, not a selected HI.
- **Collector-emitter voltage absolute peak / RMS:** may capture electrical response, but can mix switching states and condition changes. RMS can be smoother, yet smoothness alone is not sensitivity or specificity. Require calibrated units, matched operating points, and aligned events.
- **Gate-emitter voltage peak-to-peak:** potentially useful for monitoring drive response or as a control, but changes may be due to commanded gate conditions. It should not be called device health without evidence separating drive changes from device degradation.
- **Collector-emitter current RMS / peak-to-peak:** electrically relevant, but the documented drift directly threatens validity. A consistent trend could be sensor drift; resolve calibration before using it as HI.
- **Gate-command peak-to-peak:** best treated as experimental context / negative control. If the applied signal changes with time, it flags a confound; absent a specific mechanism, it is not a degradation HI.

**No automatic winner:** voltage peak-to-peak may deserve follow-up because of physical plausibility, while current has an explicit data-quality warning and gate-command voltage is primarily context. None meets the evidence bar for a validated HI in this dataset. The next decision should be to define the mechanism and collect/align the evidence needed to test it.

## 7. What would validate a candidate HI?

Before promotion from feature to HI, plan a study that:

1. States the degradation mechanism and expected feature direction before looking for a favorable trend.
2. Defines aging progression from verified cycles, dose, inspections, or failure events—not filenames or capture timestamps alone.
3. Aligns comparable switching events and controls/records gate drive, supply, temperature, load, `dt`, capture window, and measurement bandwidth.
4. Resolves sensor calibration and the documented current drift; includes repeated measurements and healthy baselines to quantify repeatability.
5. Tests monotonicity, trend, noise, sensitivity, and robustness within each device, then confirms the relationship on held-out devices or a separate experiment.
6. Predefines an endpoint and threshold only when supported by engineering evidence; validates warning lead time and false-alarm behavior.

The transient feature CSV has no physical failure label or EOL definition. Therefore this phase does not produce a degradation classifier, threshold, composite HI, or RUL target.

## Phase 6 learning check

1. Why is a feature with high correlation to recorded time not automatically a good HI?
2. Which candidate has the clearest mechanism-based rationale, and what is still wrong with its current measurement definition?
3. What does a downward current trend mean given the documented drift warning?
4. Why compare trends separately by device instead of pooling all captures?
5. How do monotonicity, noise, stability, and sensitivity describe different properties?
6. What independent measurements and controls would you add before claiming that one candidate tracks degradation?

Run the notebook in order and use the per-device results to support your answers. A justified conclusion may be that the evidence is insufficient to select an HI.